In [1]:
import matplotlib.pyplot as plt
import numpy as np
import random
#from diffuser.eqnet_independent import EqNet, GaussianDiffusion, DiffuserTrainer, DiffuserPlanner, CompositeReward, expand_spline_from_skip_list, StartReachingReward, GoalReachingReward, SkipTotalTimeSkipPenalty, CurvaturePenalty, LogSkipReward
from timeskip_diffuser.diffuser.nets import EqNet, TemporalUNet
from timeskip_diffuser.diffuser.diffusion import GaussianDiffusion
from timeskip_diffuser.diffuser.trainer import DiffuserTrainer
from timeskip_diffuser.diffuser.planner import DiffuserPlanner, expand_spline_from_skip_list
from timeskip_diffuser.diffuser.reward import CompositeReward, StartReachingReward, GoalReachingReward, TotalTimeSkipPenalty, CurvaturePenalty, LogSkipReward


import torch
from matplotlib.patches import Rectangle
import mujoco
from torch.utils.data import DataLoader, Dataset
from timeskip_diffuser.datasets.point_maze.offline_skip.traj_verifiers.verifier import extract_wall_rects, verify_trajectory_dense, sample_free_point, endpoint_within_eps
from timeskip_diffuser.datasets.point_maze.offline_skip.offline_skip import OfflineSkipDataset
import json
    
from timeskip_diffuser.datasets.point_maze.umaze import UMazeFlatDataset



def run_planning_experiment_with_viz(
    planner,
    dataset_name,
    starts,
    goals,
    max_tries=10,
    horizon=32,
    seed=0,
    guidance_scale=1.0,
):
    num_tasks = len(starts)

    wall_rects = extract_wall_rects(dataset_name)

    # Hard-coded safe bounds for U-Maze
    bounds = (-2.8, 2.8, -2.8, 2.8)
    xmin, xmax, ymin, ymax = bounds
    start_goal_pairs = [] #for saving later
    successes = 0

    for task_id in range(num_tasks):
        print(f"\n=== Task {task_id + 1}/{num_tasks} ===")

        # --------------------------------------------------
        # Create a NEW figure per task  ← THIS IS CRITICAL
        # --------------------------------------------------

        # --------------------------------------------------
        # Sample start & goal
        # --------------------------------------------------
        start_xy = starts[task_id]
        goal_xy  = goals[task_id]
        start_goal_pairs.append([
            [float(start_xy[0]), float(start_xy[1])],
            [float(goal_xy[0]),  float(goal_xy[1])],
        ])
        print(f"Start: {start_xy}, Goal: {goal_xy}")


        current = np.array([start_xy[0], start_xy[1], 0.0, 0.0], dtype=np.float32)
        goal = goal_xy

        reward_fn = CompositeReward(
            [
                StartReachingReward(start_xy, reward_scale=5.0),
                GoalReachingReward(goal, reward_scale=5.0),
                CurvaturePenalty(reward_scale=0.05),
                TotalTimeSkipPenalty(reward_scale=0.01)
            ]
        )

        solved = False

        # --------------------------------------------------
        # Attempt loop — STOP AFTER FIRST SUCCESS
        # --------------------------------------------------
        print("Current", current)
        print("Goal", goal)
        for attempt in range(max_tries):
            traj = planner.plan(
                current[:2],
                goal,
                reward_fn=reward_fn,
                horizon=horizon,
                guidance_scale=guidance_scale,
                condition_on_start=True,
                condition_on_goal=True,
                conditioning_schedule="constant",
                conditioning_strength=0.9,
            )

            pos_dense = traj
            feasible, coll_idx = verify_trajectory_dense(pos_dense, wall_rects) #no collision
            start_ok, goal_ok, start_err, goal_err = endpoint_within_eps(
                pos_dense,
                start_xy,
                goal_xy,
                0.20,
                0.20,
            ) #start,end is within reasonable dist of targets

            success = feasible and start_ok and goal_ok
            
            if success:
                dataset.visualize(pos_dense)
                print(
                    f"  ✓ Attempt {attempt + 1} succeeded "
                    f"(start_err={start_err:.3f}, goal_err={goal_err:.3f})"
                )
                successes += 1
                solved = True
                break
            else:
                dataset.visualize(pos_dense)

                print(
                    f"  ✗ Attempt {attempt + 1} failed "
                    f"(collision={not feasible}, "
                    f"start_err={start_err:.3f}, goal_err={goal_err:.3f})"
                )

        # --------------------------------------------------
        # Task summary
        # --------------------------------------------------
        if solved:
            print("✓ Task solved")
        else:
            print("✗ Task failed (all attempts)")

    
    with open("start_goal_pairs_u_maze.json", "w") as f:
        json.dump(start_goal_pairs, f, indent=2)
    
    # --------------------------------------------------
    # Final summary
    # --------------------------------------------------
    print("\n==============================")
    print(f"Total tasks  : {num_tasks}")
    print(f"Max tries   : {max_tries}")
    print(f"Successes   : {successes}")
    print(f"Success rate: {successes / num_tasks:.3f}")
    print("==============================")

    return successes






device = "cuda" if torch.cuda.is_available() else "cpu"

#Load dataset that the model was trained on for denormalization statistics
"""
OFFLINE_FILE = "/scratch/network/ts4953/recloned/rpmml-project/timeskip-diffuser/src/timeskip_diffuser/datasets/offline_datasets/medium_h32_mu1_sig1.npz"
path = "/scratch/network/ts4953/recloned/rpmml-project/timeskip-diffuser/src/timeskip_diffuser/datasets/offline_datasets/medium_h32_mu1_sig1.npz"
archive = np.load(path)
"""



TASKS_FILE = "/scratch/network/ts4953/recloned/rpmml-project/timeskip-diffuser/src/timeskip_diffuser/datasets/point_maze/start_goal/umaze_start_goal_100.npz"
task_archive = np.load(TASKS_FILE)

starts = task_archive["starts"]  # (N,2)
goals  = task_archive["goals"]   # (N,2)
assert len(starts) == len(goals)

for k in archive.files:
    arr = archive[k]
    print(f"{k:12s} | shape={arr.shape}, dtype={arr.dtype}")
    

print("flat_mean:", archive["flat_mean"])
print("flat_std :", archive["flat_std"])
print("skip_mean:", archive["skip_mean"].item())
print("skip_std :", archive["skip_std"].item())
print("full_mean:", archive["full_mean"])
print("full_std :", archive["full_std"])

#dataset = OfflineSkipDataset(OFFLINE_FILE, horizon=32)
dataset = UMazeFlatDataset(horizon=32)

#Load model + create planner
eqnet = EqNet(
       state_dim=dataset.state_dim,
       hidden_dim=128,
       time_dim=32,
       n_layers=10,
   )


diffusion = GaussianDiffusion(timesteps=200)
trainer = DiffuserTrainer(
       model = eqnet,
       diffusion = diffusion,
       dataset=dataset,
       device = device)
trainer.use_ema_for_inference()


trainer.load_checkpoint("/scratch/network/ts4953/recloned/rpmml-project/timeskip-diffuser/src/timeskip_diffuser/pipelines/runs/umaze_eqnet_flat_seed0_20251221_033052/checkpoints/diffuser_eqnet_epoch_20.pt")
planner = DiffuserPlanner(eqnet, diffusion, dataset, device=device)


run_planning_experiment_with_viz(
    planner,
    dataset_name="D4RL/pointmaze/umaze-v2",
    starts=starts,
    goals=goals,
    max_tries=10,
    horizon=32,
)



NameError: name 'archive' is not defined

In [9]:
#READ FROM NPZ
import numpy as np

NPZ_PATH = "/scratch/network/ts4953/recloned/rpmml-project/timeskip-diffuser/src/timeskip_diffuser/datasets/point_maze/start_goal/medium_start_goal_100.npz"

archive = np.load(NPZ_PATH)

# List keys
print("=== Keys in NPZ ===")
for k in archive.files:
    print(f"{k:15s} | shape={archive[k].shape}, dtype={archive[k].dtype}")

# Print starts and goals
starts = archive["starts"]
goals  = archive["goals"]

print("\n=== Starts ===")
print(starts)          # (100, 2)

print("\n=== Goals ===")
print(goals)           # (100, 2)


=== Keys in NPZ ===
starts          | shape=(100, 2), dtype=float32
goals           | shape=(100, 2), dtype=float32

=== Starts ===
[[ 1.0975914e+00 -4.6113107e-01]
 [-1.1402848e+00 -3.2110450e-01]
 [-2.7061217e+00  1.6058695e+00]
 [ 1.1158311e-01  1.6254059e+00]
 [-2.1351070e+00  1.1358826e+00]
 [-2.4209661e+00  2.8473051e+00]
 [ 2.2651579e+00 -2.0918891e+00]
 [-8.6774284e-01  6.0480756e-01]
 [ 2.5500093e+00 -1.2981976e+00]
 [-8.7118633e-02 -1.2257425e+00]
 [ 2.2552280e+00 -2.0667243e-01]
 [ 2.6255183e+00  2.1783366e+00]
 [ 1.6116691e+00 -2.5253205e+00]
 [-4.2291924e-01 -1.2625968e+00]
 [ 2.1509304e+00 -1.7134390e+00]
 [ 9.0957147e-01  1.3796707e+00]
 [ 1.6820885e-01 -3.4926394e-01]
 [ 2.4086070e+00  1.8689538e+00]
 [ 1.2666330e+00  2.7771628e+00]
 [-1.1029072e+00  1.9256836e+00]
 [ 2.2553766e+00 -4.7488371e-01]
 [-1.8843076e+00  6.2057680e-01]
 [ 1.9309763e+00  2.3391078e+00]
 [-1.9102945e+00 -8.2232499e-01]
 [-2.5285406e+00  2.0535290e+00]
 [ 4.7187528e-01 -1.8793759e+00]
 [ 1.60955